# Module 2: Banking Transformations

**Objective**: Learn practical data transformations for banking analytics.

## What You'll Learn
1. Multi-table joins (Customers ↔ Accounts ↔ Transactions)
2. Customer segmentation logic
3. Aggregations and grouping
4. Creating derived columns with business logic

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module02").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
customers_df = spark.read.parquet(f"{S3_RAW}/customers") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "customers.csv"), header=True, inferSchema=True)
accounts_df = spark.read.parquet(f"{S3_RAW}/accounts") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)
transactions_df = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
branches_df = spark.read.parquet(f"{S3_RAW}/branches") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "branches.csv"), header=True, inferSchema=True)

print(f"Loaded: {customers_df.count()} customers, {accounts_df.count()} accounts, {transactions_df.count()} transactions")

## Multi-Table Joins

In [ ]:
# Join customers with accounts
customer_accounts = customers_df.join(accounts_df, "customer_id", "inner")
customer_accounts.select("customer_id", "name", "segment", "account_id", "balance").show(5)

## Customer Aggregations

In [ ]:
# Customer summary
customer_summary = accounts_df.groupBy("customer_id").agg(
    count("account_id").alias("account_count"),
    spark_round(sum("balance"), 2).alias("total_balance"),
    spark_round(avg("balance"), 2).alias("avg_balance")
)
customer_summary.orderBy(col("total_balance").desc()).show(10)

## Customer Segmentation Logic

In [ ]:
# Re-segment based on balance
customer_profile = customers_df.join(customer_summary, "customer_id", "left").fillna({"total_balance": 0})
customer_resegment = customer_profile.withColumn(
    "calculated_segment",
    when(col("total_balance") >= 5_000_000_000, "UHNW")
    .when(col("total_balance") >= 500_000_000, "HNW")
    .when(col("total_balance") >= 100_000_000, "Affluent")
    .when(col("total_balance") >= 20_000_000, "Mass Affluent")
    .otherwise("Mass")
)
customer_resegment.select("name", "segment", "calculated_segment", "total_balance").show(10)

## Transaction Analysis

In [ ]:
# Monthly by channel
txn_parsed = transactions_df.withColumn("txn_month", date_format(to_date(col("txn_datetime")), "yyyy-MM"))
monthly_by_channel = txn_parsed.groupBy("txn_month", "channel").agg(count("*").alias("count"), sum("amount").alias("total"))
monthly_by_channel.orderBy("txn_month").show(15)

In [ ]:
# Transaction categorization
txn_cat = transactions_df.withColumn("size",
    when(col("amount") >= 100_000_000, "Large").when(col("amount") >= 10_000_000, "Medium").otherwise("Small")
)
txn_cat.groupBy("size").count().show()

## Practice Exercises
1. Find top 10 branches by transaction volume
2. Calculate deposit-to-withdrawal ratio per customer
3. Find customers with no transactions (left_anti join)

In [ ]:
spark.stop()